# 🤖 AI-Powered Inventory Optimization & Demand Forecasting
## ☁️ Google Colab — One-Click Deployment

[![FastAPI](https://img.shields.io/badge/FastAPI-0.109+-green.svg)](https://fastapi.tiangolo.com)
[![React](https://img.shields.io/badge/React-18.2+-61dafb.svg)](https://react.dev)
[![Python](https://img.shields.io/badge/Python-3.9+-blue.svg)](https://python.org)
[![License](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)

---

### 🚀 Quick Start (2 steps)

| Step | Action |
|:----:|--------|
| **1** | Click **Runtime → Run all** (or press `Ctrl + F9`) |
| **2** | Wait ~5 min → click the **public URL** displayed in the final cell |

> 💡 **No local setup required** — everything runs in the cloud.
>
> If using **localtunnel**, you'll need to paste the displayed **IP address** as a password on the tunnel page.

---

In [ ]:
#@title ⚙️ Configuration (UPDATE BEFORE RUNNING) { display-mode: "form" }

#@markdown ### 📂 GitHub Repository URL
REPO_URL = "https://github.com/vempalamohit-bot/AI-Powered-Inventory-Optimization-Demand-Forecasting.git" #@param {type:"string"}

#@markdown ---
#@markdown ### 🔑 API Keys *(Optional — app works without these)*
OPENAI_API_KEY = "" #@param {type:"string"}
SENDGRID_API_KEY = "" #@param {type:"string"}
ALERT_EMAIL = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Advanced
PORT = 8000 #@param {type:"integer"}
TUNNEL = "localtunnel" #@param ["localtunnel", "cloudflared"]

In [ ]:
#@title 📦 Step 1/4 — Install Dependencies & Clone Repository { display-mode: "form" }
import os, subprocess, sys, shutil

PROJECT = '/content/Miracle_AI_driven_POC'
BACKEND = f'{PROJECT}/backend'
FRONTEND = f'{PROJECT}/frontend'

print("━" * 60)
print("📦 STEP 1 — Install dependencies & clone repository")
print("━" * 60)

# ── 1. Install Node.js 20 LTS (Vite 5 requires Node >= 18) ────
print("\n[1/5] 🔧 Installing Node.js 20 LTS...")

# Remove old Node.js completely
subprocess.run('apt-get remove -y nodejs npm > /dev/null 2>&1', shell=True)
subprocess.run('rm -rf /usr/local/lib/node_modules /usr/local/bin/node /usr/local/bin/npm /usr/local/bin/npx', shell=True)

# Install Node.js 20 via NodeSource (two-step for reliability)
r1 = subprocess.run(
    'curl -fsSL https://deb.nodesource.com/setup_20.x -o /tmp/nodesource_setup.sh',
    shell=True, capture_output=True, text=True
)
r2 = subprocess.run('bash /tmp/nodesource_setup.sh', shell=True, capture_output=True, text=True)

if r2.returncode != 0:
    # Fallback: install via n (node version manager)
    print("  ⚠️  NodeSource failed, using fallback installer...")
    subprocess.run('apt-get update -qq > /dev/null 2>&1', shell=True)
    subprocess.run('apt-get install -y -qq nodejs npm > /dev/null 2>&1', shell=True)
    subprocess.run('npm install -g n > /dev/null 2>&1', shell=True)
    subprocess.run('n 20 > /dev/null 2>&1', shell=True)
    os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']
else:
    subprocess.run('apt-get install -y -qq nodejs > /dev/null 2>&1', shell=True)

# Install localtunnel
subprocess.run('npm i -g localtunnel > /dev/null 2>&1', shell=True)

# Verify Node version
node_v = subprocess.run('node -v', shell=True, capture_output=True, text=True).stdout.strip()
npm_v = subprocess.run('npm -v', shell=True, capture_output=True, text=True).stdout.strip()
print(f"  ✅ Node.js {node_v}  |  npm {npm_v}  |  localtunnel installed")

# Check if Node version is sufficient
try:
    major = int(node_v.lstrip('v').split('.')[0])
    if major < 18:
        print(f"  ⚠️  Node {node_v} is too old for Vite 5. Upgrading via n...")
        subprocess.run('npm install -g n > /dev/null 2>&1', shell=True)
        subprocess.run('n 20 > /dev/null 2>&1', shell=True)
        os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']
        node_v = subprocess.run('/usr/local/bin/node -v', shell=True, capture_output=True, text=True).stdout.strip()
        print(f"  ✅ Upgraded to Node.js {node_v}")
except Exception:
    pass

# ── 2. Clone repository ────────────────────────────────────────
print("\n[2/5] 📂 Cloning repository...")
if "YOUR_USERNAME" in REPO_URL:
    raise RuntimeError(
        "❌ Please update REPO_URL in the Configuration cell (Cell 2) above!\n"
        "   Example: https://github.com/johndoe/Miracle_AI_driven_POC.git"
    )

if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT)

r = subprocess.run(
    f'git clone --depth 1 {REPO_URL} {PROJECT}',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print(f"  ❌ Clone failed:\n{r.stderr}")
    raise RuntimeError("Clone failed — check your REPO_URL and make sure the repo is public (or use a token)")
print(f"  ✅ Repository cloned to {PROJECT}")

# ── 3. Fix numpy (Colab pre-installed numpy may conflict) ──────
print("\n[3/5] 🔧 Fixing numpy compatibility...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy<2.0.0'],
    capture_output=True
)
print("  ✅ numpy compatible version installed")

# ── 4. Python packages ─────────────────────────────────────────
print("\n[4/5] 🐍 Installing Python packages (2-3 min)...")
r = subprocess.run(
    f'{sys.executable} -m pip install -q --no-deps -r {BACKEND}/requirements.txt',
    shell=True, capture_output=True, text=True
)
# Now install with deps but skip numpy (already fixed)
r = subprocess.run(
    f'{sys.executable} -m pip install -q -r {BACKEND}/requirements.txt',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    failed = [l for l in r.stderr.split('\n') if 'ERROR' in l]
    if failed:
        print(f"  ⚠️  Some optional packages had issues (app will still work):")
        for f_line in failed[:3]:
            print(f"     {f_line.strip()[:80]}")
print("  ✅ Python packages installed")

# ── 5. Frontend packages ───────────────────────────────────────
print("\n[5/5] ⚛️  Installing frontend packages...")
subprocess.run(
    f'cd {FRONTEND} && npm install --silent 2>/dev/null',
    shell=True, capture_output=True
)
print("  ✅ Frontend packages installed")

print("\n" + "━" * 60)
print("✅ STEP 1 COMPLETE — All dependencies ready")
print("━" * 60)

In [ ]:
#@title 🗄️ Step 2/4 — Setup Database { display-mode: "form" }
import os, sys, time, subprocess

PROJECT = '/content/Miracle_AI_driven_POC'
BACKEND = f'{PROJECT}/backend'
DB = f'{BACKEND}/inventory.db'

print("━" * 60)
print("🗄️  STEP 2 — Building database from CSV data")
print("━" * 60)

# Remove existing DB to start fresh
for ext in ['', '-wal', '-shm', '-journal']:
    p = DB + ext
    if os.path.exists(p):
        os.remove(p)

# ── Create a standalone database loader script ──────────────────
# (Runs in a fresh subprocess to avoid Colab's numpy cache issue)
loader_script = f'''\
import os, sys, time, sqlite3
sys.path.insert(0, "{BACKEND}")

# Fix numpy if needed
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy<2.0.0'], capture_output=True)

import pandas as pd

DB = "{DB}"
PRODUCTS_CSV = "{PROJECT}/data/products_50k.csv"
SALES_CSV = "{PROJECT}/data/sales_dense.csv"

# Verify files
for path, label in [(PRODUCTS_CSV, "products_50k.csv"), (SALES_CSV, "sales_dense.csv")]:
    if not os.path.exists(path):
        print(f"ERROR: {{label}} not found at {{path}}")
        sys.exit(1)
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"  Found {{label}} ({{size:.1f}} MB)")

# Create tables via SQLAlchemy models
from app.database import engine, Base
from app.database.models import Product, SalesHistory
Base.metadata.create_all(bind=engine)
print("  Tables created")

t0 = time.time()
conn = sqlite3.connect(DB)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=OFF")
conn.execute("PRAGMA cache_size=-64000")
conn.execute("PRAGMA temp_store=MEMORY")

# Helper: get actual table columns so we only insert matching CSV columns
def get_table_columns(conn, table_name):
    cursor = conn.execute(f"PRAGMA table_info({{table_name}})")
    return [row[1] for row in cursor.fetchall()]

# Load products
print("  Loading products...")
products_df = pd.read_csv(PRODUCTS_CSV)
products_df.insert(0, "id", range(1, len(products_df) + 1))
# Filter to only columns that exist in the products table
product_cols = get_table_columns(conn, "products")
extra_cols = [c for c in products_df.columns if c not in product_cols]
if extra_cols:
    print(f"  Dropping CSV-only columns: {{extra_cols}}")
    products_df = products_df[[c for c in products_df.columns if c in product_cols]]
products_df.to_sql("products", conn, if_exists="append", index=False, chunksize=5000)
pc = conn.execute("SELECT COUNT(*) FROM products").fetchone()[0]
print(f"  Products: {{pc:,}}")

# Load sales
print("  Loading sales (30-60s)...")
sales_df = pd.read_csv(SALES_CSV)
sales_df.insert(0, "id", range(1, len(sales_df) + 1))
# Filter to only columns that exist in the sales_history table
sales_cols = get_table_columns(conn, "sales_history")
extra_sales = [c for c in sales_df.columns if c not in sales_cols]
if extra_sales:
    print(f"  Dropping CSV-only columns: {{extra_sales}}")
    sales_df = sales_df[[c for c in sales_df.columns if c in sales_cols]]
sales_df.to_sql("sales_history", conn, if_exists="append", index=False, chunksize=10000)
sc = conn.execute("SELECT COUNT(*) FROM sales_history").fetchone()[0]
print(f"  Sales: {{sc:,}}")

# Indexes
conn.execute("CREATE INDEX IF NOT EXISTS ix_products_id ON products (id)")
conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS ix_products_sku ON products (sku)")
conn.execute("CREATE INDEX IF NOT EXISTS ix_products_name ON products (name)")
conn.execute("CREATE INDEX IF NOT EXISTS ix_products_category ON products (category)")
conn.execute("CREATE INDEX IF NOT EXISTS ix_sales_history_id ON sales_history (id)")
conn.execute("CREATE INDEX IF NOT EXISTS ix_sales_history_date ON sales_history (date)")
conn.execute("CREATE INDEX IF NOT EXISTS ix_sales_product_id ON sales_history (product_id)")
conn.commit()
conn.execute("PRAGMA optimize")
conn.close()

elapsed = time.time() - t0
db_size = os.path.getsize(DB) / (1024 * 1024)
print(f"DONE|{{pc}}|{{sc}}|{{db_size:.1f}}|{{elapsed:.0f}}")
'''

loader_path = f'{BACKEND}/_colab_db_loader.py'
with open(loader_path, 'w') as f:
    f.write(loader_script)

# ── Run in a fresh subprocess (avoids numpy cache corruption) ──
print("\n⏳ Loading 50K products + 2M sales records...\n")
t0 = time.time()
r = subprocess.run(
    [sys.executable, loader_path],
    capture_output=True, text=True, timeout=300
)

# Show output
for line in r.stdout.strip().split('\n'):
    if line.strip():
        print(f"  {line}")

if r.returncode != 0:
    print(f"\n❌ Database setup failed:")
    print(r.stderr[-1000:])
    raise RuntimeError("Database setup failed")

# Parse results from last line
parts = r.stdout.strip().split('\n')[-1]
if parts.startswith('DONE|'):
    vals = parts.split('|')
    products_count = int(vals[1])
    sales_count = int(vals[2])
    db_size = float(vals[3])
    elapsed = float(vals[4])
else:
    # Fallback: check DB directly
    import sqlite3
    conn = sqlite3.connect(DB)
    products_count = conn.execute("SELECT COUNT(*) FROM products").fetchone()[0]
    sales_count = conn.execute("SELECT COUNT(*) FROM sales_history").fetchone()[0]
    db_size = os.path.getsize(DB) / (1024 * 1024)
    elapsed = time.time() - t0
    conn.close()

# Cleanup
os.remove(loader_path)

print("\n" + "━" * 60)
print(f"✅ STEP 2 COMPLETE — Database ready!")
print(f"   📦 Products: {products_count:,}  |  📊 Sales: {sales_count:,}")
print(f"   💾 Size: {db_size:.1f} MB  |  ⏱ Time: {elapsed:.0f}s")
print("━" * 60)

if products_count == 0 or sales_count == 0:
    raise RuntimeError(f"❌ Database verification failed: {products_count} products, {sales_count} sales")

In [ ]:
#@title 🏗️ Step 3/4 — Build Frontend & Prepare Server { display-mode: "form" }
import os, subprocess

PROJECT = '/content/Miracle_AI_driven_POC'
FRONTEND = f'{PROJECT}/frontend'
BACKEND = f'{PROJECT}/backend'

print("━" * 60)
print("🏗️  STEP 3 — Build frontend & prepare unified server")
print("━" * 60)

# ── 1. Inject same-origin API base URL ─────────────────────────
print("\n[1/3] 🔗 Configuring API endpoint...")
index_html = f'{FRONTEND}/index.html'
with open(index_html) as f:
    html = f.read()

if 'window.__API_BASE__' not in html:
    html = html.replace(
        '<div id="root">',
        '<script>window.__API_BASE__="/api";</script>\n    <div id="root">'
    )
    with open(index_html, 'w') as f:
        f.write(html)
print("  ✅ API endpoint configured (same-origin /api)")

# ── 2. Build React frontend ───────────────────────────────────
print("\n[2/3] ⚛️  Building React frontend (~60s)...")
r = subprocess.run(
    'npm run build', shell=True, cwd=FRONTEND,
    capture_output=True, text=True
)
dist_dir = f'{FRONTEND}/dist'
if r.returncode == 0 and os.path.isdir(dist_dir):
    dist_size = sum(
        os.path.getsize(os.path.join(dp, fn))
        for dp, _, fns in os.walk(dist_dir) for fn in fns
    ) / 1024
    print(f"  ✅ Frontend built ({dist_size:.0f} KB)")
else:
    print("  ⚠️  Frontend build failed — API + Swagger docs still available at /docs")
    if r.stderr:
        print(f"     {r.stderr[:200]}")

# ── 3. Create unified Colab server (API + SPA on one port) ────
print("\n[3/3] 🔧 Creating unified Colab server...")
colab_server_code = '''\
"""
Unified Colab server — Serves FastAPI backend + React frontend on one port.
API routes (/api/*) are handled first; all other paths serve the React SPA.
"""
import os, sys

_dir = os.path.dirname(os.path.abspath(__file__))
if _dir not in sys.path:
    sys.path.insert(0, _dir)

# Disable alert scheduler in Colab to prevent email spam
os.environ.setdefault("ALERT_SCHEDULER_ENABLED", "false")

from main import app  # noqa: E402

# ── Mount React production build alongside the API ──────────────
DIST = os.path.normpath(os.path.join(_dir, "..", "frontend", "dist"))

if os.path.isdir(DIST):
    from fastapi.staticfiles import StaticFiles
    from fastapi.responses import FileResponse
    from starlette.middleware.base import BaseHTTPMiddleware

    _index = os.path.join(DIST, "index.html")

    # Remove the JSON root endpoint so React app loads at /
    app.routes[:] = [r for r in app.routes
                     if not (hasattr(r, "path") and r.path == "/"
                             and hasattr(r, "methods"))]

    class _SPAFallback(BaseHTTPMiddleware):
        """Serve index.html for client-side routes (React Router).
        Skips API/docs paths and file-like requests (with extensions)."""
        async def dispatch(self, request, call_next):
            p = request.url.path
            # Root path — always serve React dashboard
            if p == "/" and os.path.exists(_index):
                return FileResponse(_index, media_type="text/html")
            resp = await call_next(request)
            if (
                resp.status_code == 404
                and not p.startswith(("/api", "/docs", "/redoc", "/openapi"))
                and "." not in p.rsplit("/", 1)[-1]
            ):
                return FileResponse(_index, media_type="text/html")
            return resp

    app.add_middleware(_SPAFallback)
    app.mount("/", StaticFiles(directory=DIST, html=True), name="spa")
    print("✅ Serving: API (/api) + React frontend (/)")
else:
    print("⚠️  frontend/dist not found — serving API only (/docs for Swagger)")
'''

with open(f'{BACKEND}/colab_server.py', 'w') as f:
    f.write(colab_server_code)
print("  ✅ Colab server created (backend/colab_server.py)")

print("\n" + "━" * 60)
print("✅ STEP 3 COMPLETE — Application ready to launch!")
print("━" * 60)

In [ ]:
#@title 🚀 Step 4/4 — Launch Application { display-mode: "form" }
import subprocess, time, os, sys, re, urllib.request

PROJECT = '/content/Miracle_AI_driven_POC'
BACKEND = f'{PROJECT}/backend'

print("━" * 60)
print("🚀 STEP 4 — Launching application")
print("━" * 60)

# ── 1. Set environment variables ──────────────────────────────
os.environ['ALERT_SCHEDULER_ENABLED'] = 'false'
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    os.environ['GENAI_ENABLED'] = 'true'
if SENDGRID_API_KEY:
    os.environ['SENDGRID_API_KEY'] = SENDGRID_API_KEY
if ALERT_EMAIL:
    os.environ['ALERT_RECIPIENTS'] = ALERT_EMAIL

# ── 2. Start unified server (API + Frontend) ─────────────────
print("\n🖥️  Starting server on port", PORT, "...")
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'colab_server:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=BACKEND,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
# Wait for server to become ready
for attempt in range(15):
    time.sleep(2)
    try:
        resp = urllib.request.urlopen(f'http://localhost:{PORT}/')
        if resp.status == 200:
            print("  ✅ Server is running")
            break
    except Exception:
        if attempt == 14:
            print("  ⚠️  Server may still be starting — continuing anyway...")

# ── 3. Get public IP (localtunnel password) ───────────────────
print("\n🔑 Getting public IP address...")
ip = "unavailable"
for ip_service in ['https://api.ipify.org', 'https://ipinfo.io/ip', 'https://ifconfig.me']:
    try:
        ip = urllib.request.urlopen(ip_service, timeout=5).read().decode().strip()
        break
    except Exception:
        continue
print(f"  ✅ IP: {ip}")

# ── 4. Create public tunnel ──────────────────────────────────
print(f"\n🌐 Creating {TUNNEL} tunnel...")
tunnel_url = None

if TUNNEL == "localtunnel":
    # ── localtunnel: free, no signup, requires IP as password ──
    lt_out = '/tmp/lt_output.txt'
    open(lt_out, 'w').close()  # clear file
    subprocess.Popen(f'lt --port {PORT} > {lt_out} 2>&1', shell=True)
    for _ in range(20):
        time.sleep(1)
        try:
            with open(lt_out) as f:
                content = f.read()
                match = re.search(r'https://[^\s]+\.loca\.lt', content)
                if match:
                    tunnel_url = match.group()
                    break
        except Exception:
            pass

elif TUNNEL == "cloudflared":
    # ── Cloudflare tunnel: free, no signup, no password needed ──
    print("  ⬇️  Downloading cloudflared...")
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
        '-O /tmp/cloudflared && chmod +x /tmp/cloudflared',
        shell=True, capture_output=True
    )
    cf_out = '/tmp/cf_output.txt'
    open(cf_out, 'w').close()
    subprocess.Popen(
        f'/tmp/cloudflared tunnel --url http://localhost:{PORT} 2>&1 | tee {cf_out}',
        shell=True
    )
    for _ in range(25):
        time.sleep(1)
        try:
            with open(cf_out) as f:
                content = f.read()
                match = re.search(r'https://[^\s]+\.trycloudflare\.com', content)
                if match:
                    tunnel_url = match.group()
                    break
        except Exception:
            pass

# ── 5. Display access info ────────────────────────────────────
print("\n" + "━" * 60)
if tunnel_url:
    print("🎉 YOUR APPLICATION IS LIVE!")
    print("━" * 60)
    print()
    print(f"  🔗 App URL:       {tunnel_url}")
    print(f"  📖 API Docs:      {tunnel_url}/docs")
    print(f"  📊 Dashboard:     {tunnel_url}/")
    print(f"  📦 Products:      {tunnel_url}/products")
    print(f"  📈 Forecasting:   {tunnel_url}/forecasting")
    print()
    if TUNNEL == "localtunnel":
        print(f"  ┌─────────────────────────────────────────────┐")
        print(f"  │  🔑 Tunnel Password (IP): {ip:<18s} │")
        print(f"  └─────────────────────────────────────────────┘")
        print()
        print("  📋 How to access:")
        print("     1. Click the App URL above")
        print("     2. On the tunnel page, paste this IP:", ip)
        print("     3. Click 'Click to Submit'")
        print("     4. The full application loads! 🎉")
    else:
        print("  📋 Click the App URL above — no password needed!")
    print()
    print("  ⏰ Session timeout: ~90 min (Colab free tier)")
    print("  🔄 To restart: Runtime → Run all")
else:
    print("⚠️  Could not capture tunnel URL")
    print("━" * 60)
    print()
    print("  Try manually in a new cell:")
    print(f"     !lt --port {PORT}")
    print()
    print(f"  Or access API docs directly (Colab only):")
    print(f"     http://localhost:{PORT}/docs")
print("━" * 60)

# Keep the cell running so the server stays alive
print("\n⏳ Server is running. Do NOT close this cell.")
print("   To stop: Runtime → Interrupt execution")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n🛑 Server stopped.")
    server.terminate()

---
## 🛠 Troubleshooting

| Issue | Solution |
|-------|----------|
| `REPO_URL` error | Update the URL in **Cell 2** with your real GitHub repo URL |
| Clone fails | Make sure the repo is **public**, or use a [personal access token](https://github.com/settings/tokens) in the URL |
| Tunnel not working | Re-run **Cell 6** or switch `TUNNEL` to `cloudflared` in **Cell 2** |
| Database error | Re-run **Cell 4** to rebuild the database |
| Frontend blank page | Wait 10s and refresh — the server may still be starting |
| Session timeout | Colab free tier disconnects after ~90 min — just click **Runtime → Run all** again |
| Prophet install fails | This is optional — the app works fine without it |

### 📌 Architecture (what runs in Colab)

```
┌──────────────────────────────────────────────────────┐
│  Google Colab Runtime                                │
│                                                      │
│  ┌─────────────────────────────────────────────────┐ │
│  │  Unified Server (colab_server.py) — port 8000   │ │
│  │                                                 │ │
│  │  /api/*     → FastAPI backend (50+ endpoints)   │ │
│  │  /docs      → Swagger API documentation         │ │
│  │  /*         → React frontend (SPA)              │ │
│  └─────────────────────────────────────────────────┘ │
│              ↕ tunnel (localtunnel / cloudflared)     │
└──────────────────────────────────────────────────────┘
              ↕
┌──────────────────────────────────────────────────────┐
│  Your Browser → https://xyz.loca.lt                  │
│  Full React dashboard + real-time AI features        │
└──────────────────────────────────────────────────────┘
```

### 🔄 Optional: Keep Colab Session Alive
If your session keeps timing out, run this in a **new cell**:

```python
import time
while True:
    time.sleep(30)
```